## Load All Libraries

In [1]:
#Import libraries
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import odc.stac
import pystac_client
from odc.stac import load
from pystac.client import Client

## Define the Area of Interest (AOI)
In this case it's the area of Ba region.

In [2]:
#Yalelevu AOI imported from asset

aoi = gpd.read_file("/home/jovyan/Soil moisture-Individual Project/Ba.geojson")
aoi = aoi.to_crs("EPSG:4326")
min_lon, min_lat, max_lon, max_lat = aoi.total_bounds

bbox = [min_lon, min_lat, max_lon, max_lat]

In [1]:
aoi.explore()

## Load Sentinel 1 Data
Post-flood images - 6 months after flooding from January to June of 2025.

In [4]:
catalog = "https://earth-search.aws.element84.com/v1"
client = Client.open(catalog)
datetime = "2025-01-01/2025-06-30"


In [5]:
items = client.search(
    collections=["sentinel-1-grd"],
    bbox=bbox,
    datetime=datetime,
    query={
        "sar:instrument_mode": {"eq": "IW"},
        "sat:orbit_state": {"eq": "ascending"},
        "s1:orbit_source": {"eq": "RESORB"}
    }
).item_collection()

In [6]:
items

In [7]:
# Load the VV and VH
post_ds = load(
    items,
    bands=["vv", "vh"],
    bbox=bbox,
    groupby="id"
)  

In [50]:
# -----------------------------
# POST-FLOOD MEDIAN COMPOSITE
# -----------------------------
post_vh = post_ds.vh.median(dim="time")
post_vv = post_ds.vv.median(dim="time")

# -----------------------------
# APPLY LEE FILTER
# -----------------------------
post_vh_filtered = xr.DataArray(
    lee_filter(post_vh.values, size=7),
    coords=post_vh.coords,
    dims=post_vh.dims
)

post_vv_filtered = xr.DataArray(
    lee_filter(post_vv.values, size=7),
    coords=post_vv.coords,
    dims=post_vv.dims
)

# -----------------------------
# CONVERT TO dB
# -----------------------------
post_vh_db = 10 * np.log10(post_vh_filtered.clip(min=1e-6))
post_vv_db = 10 * np.log10(post_vv_filtered.clip(min=1e-6))

# -----------------------------
# VV/VH RATIO
# -----------------------------
post_ratio = post_vv_db - post_vh_db

In [2]:
plt.figure(figsize=(10,8))

post_vh_db.plot(
    cmap="gray",
    robust=True
)

plt.title("Post-Flood Sentinel-1 VH Backscatter (dB)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

In [3]:
plt.figure(figsize=(10,8))

post_vv_db.plot(
    cmap="gray",
    robust=True
)

plt.title("Post-Flood Sentinel-1 VV Backscatter (dB)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

In [53]:
post_vh_db.rio.to_raster("post_vh_db.tif")
post_vv_db.rio.to_raster("post_vv_db.tif")

In [54]:
import rioxarray as rxr

pre_vh_db = rxr.open_rasterio("pre_vh_db.tif").squeeze()
post_vh_db = rxr.open_rasterio("post_vh_db.tif").squeeze()

pre_vv_db = rxr.open_rasterio("pre_vv_db.tif").squeeze()
post_vv_db = rxr.open_rasterio("post_vv_db.tif").squeeze()

In [62]:
valid_mask = (
    np.isfinite(pre_vh_db.values)
    &
    np.isfinite(post_vh_db.values)
)

vh_change = np.where(
    valid_mask,
    post_vh_db.values - pre_vh_db.values,
    np.nan
)

vh_change = xr.DataArray(
    vh_change,
    coords=pre_vh_db.coords,
    dims=pre_vh_db.dims
)

In [4]:
plt.figure(figsize=(10,8))

vh_change.plot(
    cmap="RdBu",
    vmin=-5,
    vmax=5
)

plt.title("VH Backscatter Change (Post - Pre)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

In [64]:
print(pre_vh_db.x.equals(post_vh_db.x))
print(pre_vh_db.y.equals(post_vh_db.y))

True
True


In [65]:
print(pre_vh_db.x.equals(post_vh_db.x))
print(pre_vh_db.y.equals(post_vh_db.y))

print(np.isnan(pre_vh_db.values).sum())
print(np.isnan(post_vh_db.values).sum())

True
True
0
976


In [8]:
# ------------------
# Median composites
# ------------------
post_vv = post_ds.vv.median(dim="time")
post_vh = post_ds.vh.median(dim="time")

In [5]:
# ------------------
# Plot results
# ------------------
# Plot side-by-side
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

post_vv.plot(
ax=ax[0],
cmap="turbo"
)

ax[0].set_title("Post-Flood VV")
post_vh.plot(
ax=ax[1],
cmap="turbo"
)

ax[1].set_title("Post-Flood VH")

plt.tight_layout()
plt.show()

## Normalize
Here, we will normalise the VH image.

In [10]:
# Pre-flood VH
post_vh = post_ds.vh.median(dim="time")
post_vh_norm = (
(post_vh - post_vh.min()) /
(post_vh.max() - post_vh.min())
)

In [6]:
#Visualize the plot
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
post_vh_norm.plot(
    cmap="turbo",
    vmin=0,
    vmax=1
)
plt.title("Normalized Pre-Flood VH (0–1)")
plt.show()

## Speckle Filtering
Applying speckle filters on the normalized image.

In [12]:
# Define the lee filter functin
import numpy as np
from scipy.ndimage import uniform_filter

def lee_filter(img, size=2):
    img = img.astype(np.float32)

    mean = uniform_filter(img, size)
    mean_sq = uniform_filter(img**2, size)

    variance = mean_sq - mean**2

    overall_var = np.nanvar(img)

    weights = variance / (variance + overall_var)

    lee = mean + weights * (img - mean)

    return lee

In [13]:
#Rename the Pre VH Norm
vh_post_lee = post_vh_norm.copy()

vh_post_lee.data = lee_filter(
    post_vh_norm.values,
    size=7
)

In [7]:
#Visualize

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

post_vh_norm.plot(
    ax=ax[0],
    cmap="turbo",
    vmin=0,
    vmax=1
)
ax[0].set_title("Original Normalized VH")

vh_post_lee.plot(
    ax=ax[1],
    cmap="turbo",
    vmin=0,
    vmax=1
)
ax[1].set_title("Lee Filtered VH")

plt.tight_layout()
plt.show()

In [15]:
# Test for Gamma filter

In [16]:
#Define the gamma filter
import numpy as np
from scipy.ndimage import uniform_filter

def gamma_map_filter(img, size=2, enl=1):
    img = img.astype(np.float32)

    mean = uniform_filter(img, size)
    mean_sq = uniform_filter(img**2, size)

    var = mean_sq - mean**2

    ci = np.sqrt(var) / (mean + 1e-12)
    cu = 1 / np.sqrt(enl)

    w = np.maximum(0, (ci**2 - cu**2) / (ci**2 + 1e-12))

    filtered = mean + w * (img - mean)

    return filtered

In [17]:
vh_post_gamma = post_vh_norm.copy()

vh_post_gamma.data = gamma_map_filter(
    post_vh_norm.values,
    size=3,
    enl=1
)

In [8]:
fig, ax = plt.subplots(1, 3, figsize=(18, 6))

post_vh_norm.plot(
    ax=ax[0],
    cmap="turbo",
    vmin=0,
    vmax=0.1
)
ax[0].set_title("Original VH")

vh_post_lee.plot(
    ax=ax[1],
    cmap="turbo",
    vmin=0,
    vmax=0.1
)
ax[1].set_title("Lee Filter")

vh_post_gamma.plot(
    ax=ax[2],
    cmap="turbo",
    vmin=0,
    vmax=0.1
)
ax[2].set_title("Gamma ")

plt.tight_layout()
plt.show()

In [19]:
print("Original:", float(post_vh_norm.std()))
print("Lee:", float(vh_post_lee.std()))
print("Gamma:", float(vh_post_gamma.std()))

Original: 0.01594859352203139
Lee: 0.01444349717348814
Gamma: 0.01484459824860096


In [9]:
# #Lets plot a Line graph
# import matplotlib.pyplot as plt

# # Middle row
# row = post_vh_norm.shape[0] // 2

# orig_profile = post_vh_norm.values[row, :]
# lee_profile = vh_post_lee.values[row, :]
# gamma_profile = vh_post_gamma.values[row, :]

# plt.figure(figsize=(12, 6))

# plt.plot(orig_profile, label="Original VH", linewidth=1)
# plt.plot(lee_profile, label="Lee Filter", linewidth=2)
# plt.plot(gamma_profile, label="Gamma Filter", linewidth=2)

# plt.xlabel("Pixel")
# plt.ylabel("Normalized VH")
# plt.title("Line Profile Comparison")
# plt.legend()
# plt.grid(True, alpha=0.3)

# plt.show()

In [12]:
#Plot the histogram
plt.figure(figsize=(10,6))

post_vh_norm.plot.hist(
    bins=50,
    alpha=0.4,
    label="Original"
)

vh_post_lee.plot.hist(
    bins=50,
    alpha=0.4,
    label="Lee"
)

vh_post_gamma.plot.hist(
    bins=50,
    alpha=0.4,
    label="Gamma"
)

plt.xlabel("Normalized VH")
plt.ylabel("Frequency")
plt.title("Histogram Comparison")
plt.legend()

plt.show()

## Calculate the Difference
Comparison of the pre-flood and post-flood Sentinel‑1 VH composites revealed distinct spatial changes associated with the flood event. Negative VH differences were observed primarily along the main river channel, likely reflecting increased open-water inundation and reduced radar backscatter. In contrast, positive VH differences dominated adjacent floodplain areas, indicating increased soil moisture and saturated vegetation following the flood. Overall, the results suggest that the flood altered surface moisture conditions across the study area, with the strongest impacts occurring near the river corridor and surrounding low-lying areas.

In [22]:
import rioxarray as rxr
vh_post_gamma.rio.to_raster("Ba_Post_Flood.tif")

In [23]:
pre = rxr.open_rasterio("Ba_Pre_Flood.tif").squeeze()
post = rxr.open_rasterio("Ba_Post_Flood.tif").squeeze()

In [24]:
print(pre.shape)
print(post.shape)

print(pre.rio.crs)
print(post.rio.crs)

print(pre.rio.resolution())
print(post.rio.resolution())

(1210, 976)
(1210, 976)
EPSG:4326
EPSG:4326
(0.0001199745808999535, -0.00010543587003838705)
(0.00011998025210083631, -0.00010542238138577917)


In [25]:
print(pre.x[0].values, pre.x[-1].values)
print(post.x[0].values, post.x[-1].values)

print(pre.y[0].values, pre.y[-1].values)
print(post.y[0].values, post.y[-1].values)

177.6080300599763 177.72500527635376
177.608026977522 177.72500772332032
-17.43524449509281 -17.56271646196922
-17.435227834296224 -17.56268349339163


In [26]:
post_match = post.rio.reproject_match(pre)

vh_diff = post_match - pre

In [13]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

vh_diff.plot(
    cmap="RdBu_r",
    robust=True
)

plt.title("Pre and Post Floood Difference (Post - Pre)")
plt.show()

In [14]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

vh_diff.plot.hist(
    bins=50,
    color="steelblue",
    edgecolor="black"
)

plt.axvline(0, color="red", linestyle="--")

plt.title("VH Difference Distribution")
plt.xlabel("VH Difference (Post - Pre)")
plt.ylabel("Frequency")

plt.show()

In [29]:
increase = (vh_diff > 0).sum()
decrease = (vh_diff < 0).sum()

total = increase + decrease

print("Increase (%):", float(increase / total * 100))
print("Decrease (%):", float(decrease / total * 100))

Increase (%): 53.01432732692047
Decrease (%): 46.98567267307953


In [30]:
threshold = 0.01

significant_increase = (vh_diff > threshold).sum()
significant_decrease = (vh_diff < -threshold).sum()

print("Significant increase:", int(significant_increase))
print("Significant decrease:", int(significant_decrease))

Significant increase: 65532
Significant decrease: 7752


In [31]:
import pandas as pd

stats = pd.DataFrame({
    "Statistic": [
        "Min",
        "Max",
        "Mean",
        "Median",
        "Std Dev"
    ],
    "Value": [
        float(vh_diff.min()),
        float(vh_diff.max()),
        float(vh_diff.mean()),
        float(vh_diff.median()),
        float(vh_diff.std())
    ]
})

stats

,Statistic,Value
0,Min,-0.111032
1,Max,0.114859
2,Mean,0.001193
3,Median,0.000335
4,Std Dev,0.005084


In [32]:
threshold = np.nanpercentile(vh_diff, 5)

flood_mask = vh_diff < threshold

print("Threshold =", threshold)

Threshold = -0.0061336433514952654


In [33]:
print(type(vh_diff))

<class 'xarray.core.dataarray.DataArray'>


In [15]:
#
plt.figure(figsize=(8,8))

flood_mask.plot(
    cmap="Blues",
    add_colorbar=False
)

plt.title("Preliminary Flood Mask")
plt.show()

In [35]:
print("Threshold =", threshold)

print("Min:", np.nanmin(vh_diff.values))
print("Max:", np.nanmax(vh_diff.values))
print("Mean:", np.nanmean(vh_diff.values))
print("Std:", np.nanstd(vh_diff.values))

Threshold = -0.0061336433514952654
Min: -0.11103201
Max: 0.11485946
Mean: 0.0011928988
Std: 0.0050855563


In [16]:
plt.figure(figsize=(8,5))
plt.hist(vh_diff.values.flatten(), bins=100)
plt.axvline(threshold, color='red', linestyle='--')
plt.title("VH Difference Histogram")
plt.show()

In [37]:
mean = np.nanmean(vh_diff.values)
std = np.nanstd(vh_diff.values)

threshold = mean - 2 * std

print(threshold)

-0.008978213765658438


In [38]:
flood_mask = vh_diff < threshold

In [17]:
# flood mask
plt.figure(figsize=(8,8))

flood_mask.plot(
    cmap="Blues",
    add_colorbar=False
)

plt.title("Preliminary Flood Mask")